# BirdCLEF 2026: Train + Validate + Submit (Colab)

This notebook runs the implemented pipeline from the repository using Colab.

It covers:
- environment setup
- data checks
- CV split + leakage checks
- baseline training (and optional alt model family)
- OOF threshold optimization
- submission schema/runtime checks

Note: Kaggle hidden test soundscapes are available only during Kaggle scoring runs.

## 0) Configuration

Edit these variables before running all cells.

In [ ]:
# User-configurable settings
REPO_DIR = '/content/birdclef-2026'  # where repo exists in Colab
DRIVE_ROOT = '/content/drive/MyDrive/birdclef-2026'

# Optional toggles
RUN_ALT_MODEL_FAMILY = True
RUN_ABLATIONS = False
RUN_AUDIO_SMOKE_TEST = True

# Optional: If you copied test soundscapes into Drive for dry-run inference, set this.
LOCAL_TEST_SOUNDSCAPE_DIR = ''  # e.g. '/content/drive/MyDrive/birdclef-2026/data/test_soundscapes_local'

print('[CONFIG] REPO_DIR=', REPO_DIR)
print('[CONFIG] DRIVE_ROOT=', DRIVE_ROOT)
print('[CONFIG] RUN_ALT_MODEL_FAMILY=', RUN_ALT_MODEL_FAMILY)
print('[CONFIG] RUN_ABLATIONS=', RUN_ABLATIONS)
print('[CONFIG] RUN_AUDIO_SMOKE_TEST=', RUN_AUDIO_SMOKE_TEST)
print('[CONFIG] LOCAL_TEST_SOUNDSCAPE_DIR=', LOCAL_TEST_SOUNDSCAPE_DIR or '<not set>')

## 1) Mount Drive and Helper Runner

In [ ]:
from google.colab import drive
import os
import subprocess
import time
from pathlib import Path

print('[STEP] Mounting Google Drive...')
drive.mount('/content/drive')
print('[OK] Drive mounted.')

def run(cmd, cwd=None):
    """Run a shell command with explicit start/end timing and streamed output."""
    print('\n' + '=' * 88)
    print('[RUN]', cmd)
    if cwd:
      print('[CWD]', cwd)
    print('=' * 88)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, cwd=cwd)
    dt = time.time() - t0
    print(f'[DONE] exit_code={p.returncode} elapsed_sec={dt:.2f}')
    if p.returncode != 0:
      raise RuntimeError(f'Command failed: {cmd}')


## 2) Repository and Dependencies

In [ ]:
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    print(f'[ERROR] Repo not found at {repo_path}')
    print('Clone/copy your repo first, then rerun this cell.')
    raise FileNotFoundError(repo_path)

print(f'[OK] Repo found: {repo_path}')
run('python --version', cwd=str(repo_path))
run('pip install -r requirements-colab.txt', cwd=str(repo_path))

## 3) Data Integrity and Smoke Checks

In [ ]:
data_root = f'{DRIVE_ROOT}/data'
raw_archive = f'{DRIVE_ROOT}/raw/kaggle-download/birdclef-2026.zip'

run(
    f'python -m src.utils.data_integrity --data-root {data_root} --raw-archive {raw_archive}',
    cwd=REPO_DIR,
)

if RUN_AUDIO_SMOKE_TEST:
    run(
        'python -m src.utils.audio_smoke_test '
        f'--train-audio-dir {data_root}/train_audio '
        f'--train-soundscapes-dir {data_root}/train_soundscapes '
        '--samples-per-dir 10',
        cwd=REPO_DIR,
    )
else:
    print('[SKIP] Audio smoke test disabled by config.')

## 4) Build CV Splits and Validate Leakage

In [ ]:
run('python -m src.training.create_cv_splits --config configs/cv_policy.yaml', cwd=REPO_DIR)
run(
    f'python -m src.training.check_fold_leakage --folds-csv {DRIVE_ROOT}/outputs/reports/folds.csv',
    cwd=REPO_DIR,
)

## 5) Train Baseline Model Family

In [ ]:
run('python -m src.training.run_baseline --config configs/baseline_colab.yaml', cwd=REPO_DIR)

## 6) Train Alternative Model Family (Optional)

In [ ]:
if RUN_ALT_MODEL_FAMILY:
    run('python -m src.training.run_baseline --config configs/baseline_alt_colab.yaml', cwd=REPO_DIR)
else:
    print('[SKIP] Alternative model family disabled by config.')

## 7) Run Ablation Matrix (Optional)

In [ ]:
if RUN_ABLATIONS:
    run('python -m src.training.run_ablations --base-config configs/baseline_colab.yaml --ablation-config configs/ablations.yaml', cwd=REPO_DIR)
else:
    print('[SKIP] Ablations disabled by config.')

## 8) Optimize Class Thresholds from OOF

In [ ]:
run(
    'python -m src.training.optimize_thresholds '
    f'--oof-csv {DRIVE_ROOT}/outputs/oof/baseline_colab_v1_oof.csv '
    f'--folds-csv {DRIVE_ROOT}/outputs/reports/folds.csv '
    f'--output-csv {DRIVE_ROOT}/outputs/reports/class_thresholds.csv',
    cwd=REPO_DIR,
)

## 9) Generate Submission Dry-Run

If `LOCAL_TEST_SOUNDSCAPE_DIR` is set and contains `.ogg` files + row_id compatibility, the notebook will run checkpoint inference.

Otherwise it writes a schema-valid zero baseline `submission.csv` for pipeline checks.

In [ ]:
sample_sub = f'{DRIVE_ROOT}/data/sample_submission.csv'
submission_out = f'{DRIVE_ROOT}/outputs/submissions/submission.csv'
checkpoint = f'{DRIVE_ROOT}/outputs/checkpoints/baseline_colab_v1_fold0.pt'

if LOCAL_TEST_SOUNDSCAPE_DIR:
    cmd = (
        'python -m src.inference.generate_submission_cpu '
        f'--sample-submission {sample_sub} '
        f'--checkpoint {checkpoint} '
        f'--soundscape-dir {LOCAL_TEST_SOUNDSCAPE_DIR} '
        f'--output {submission_out}'
    )
else:
    cmd = (
        'python -m src.inference.generate_submission_cpu '
        f'--sample-submission {sample_sub} '
        f'--output {submission_out}'
    )

run(cmd, cwd=REPO_DIR)

## 10) Validate Submission + Rules Gate + Runtime Rehearsal

In [ ]:
run(
    'python -m src.inference.validate_submission '
    f'--sample-submission {DRIVE_ROOT}/data/sample_submission.csv '
    f'--submission {DRIVE_ROOT}/outputs/submissions/submission.csv',
    cwd=REPO_DIR,
)

run('python -m src.utils.metric_parity_check', cwd=REPO_DIR)
run('python -m src.utils.determinism_check', cwd=REPO_DIR)
run('python -m src.utils.rules_gate --config configs/rules_gate.yaml', cwd=REPO_DIR)

# Runtime rehearsal runs configured submission generation and checks budget.
run('python -m src.inference.runtime_rehearsal --config configs/inference_cpu_submission.yaml --max-minutes 90', cwd=REPO_DIR)

## 11) Log Submission Attempt

In [ ]:
run(
    'python -m src.utils.submission_log '
    f'--log-csv {DRIVE_ROOT}/docs/trackers/submission_log.csv '
    '--set submission_id=LOCAL_DRY_RUN run_id=colab_pipeline config_id=baseline_colab_v1 '
    'is_final_candidate=false rules_gate_passed=true notes="colab dry run"',
    cwd=REPO_DIR,
)

## 12) Kaggle Final-Scoring Note

Final competition scoring must happen in a Kaggle Notebook environment where hidden test soundscapes are mounted.

Use this repository code there with CPU + no internet + `submission.csv` output.